# 🚀 Noah's Ark OS — v1.8.0 Sprint 1
**Chapter 2.2 — Scene Intelligence & Creator Experience**

Sprint 1 changes baked into this notebook:
- `BL-O01` T112 — Oracle scroll fix (Output → Textarea)
- `BL-U02` T111 — Mission Personnel row scroll fix
- `BL-U01` T111 — Cockpit button rework (New Scene / Start / Reset Ark / Pause)
- `BL-E03` T111+T114 — Pace Control dropdown (Digital / Human / Cinematic)
- `BL-E04` T104 — Context poisoning fix (pinned + sliding hybrid)
- `BL-E02` T103+T106+T107 — Identity stability (last_utterances anti-repetition)
- `BL-E01` T114+T106 — [[SCENE_END]] signal detection (any scene length)
- `BL-I04` T108 — None-guard on token counts

Run cells **top to bottom** with `Shift+Enter`. Requires `.env` with `GEMINI_API_KEY`.


In [ ]:
# ==========================================
# T118 : ENV_LOADER
# ==========================================
# VERSION: 1.0.0 | STATUS: Stable
# ROLE: VSCode .env Secret Loader — drop-in for google.colab.userdata
# Version Remarks: Unchanged from v1.7.1. Loads GEMINI_API_KEY from .env file.
# ------------------------------------------

import os
from pathlib import Path

try:
    from dotenv import load_dotenv
    _DOTENV_AVAILABLE = True
except ImportError:
    _DOTENV_AVAILABLE = False
    print("⚠️ T118: python-dotenv not installed. Run: pip install python-dotenv")

class EnvLoader:
    def __init__(self, env_path=".env"):
        self.env_path = Path(env_path).resolve()
        self.loaded = False
        if _DOTENV_AVAILABLE:
            if self.env_path.exists():
                load_dotenv(self.env_path, override=False)
                self.loaded = True
                print(f"✅ T118: ENV loaded → {self.env_path}")
            else:
                print(f"⚠️ T118: .env not found at {self.env_path}")

    def get(self, key: str, fallback=None):
        value = os.getenv(key, fallback)
        if value is None:
            print(f"⚠️ T118: Key '{key}' not found in .env")
        return value

    def status(self):
        print(f"\n📋 T118 ENV STATUS — {self.env_path}")
        if self.loaded and self.env_path.exists():
            with open(self.env_path) as f:
                keys = [l.split('=')[0].strip() for l in f
                        if l.strip() and not l.startswith('#') and '=' in l]
            for k in keys:
                v = os.getenv(k, '<not loaded>')
                masked = f"{v[:4]}****{v[-2:]}" if v and len(v) > 6 else "****"
                print(f"   ✓ {k} = {masked}")

# ------------------------------------------
# IPO CHECK:
# Input:   .env file path
# Process: dotenv load → os.getenv lookup
# Output:  EnvLoader instance with .get(key) interface
# ------------------------------------------


In [ ]:
# ==========================================
# T000 : BIOS
# ==========================================
# VERSION: 4.0.0 | STATUS: Evolved
# ROLE: System Boot Sequence — JupyterLab Edition
# Version Remarks: google-genai Client pattern. Returns (provider, model_name, client).
#                  No interactive input() — safe for VSCode/JupyterLab kernel.
# ------------------------------------------

def boot_sequence(env_loader):
    from google import genai
    print("--- NOAH'S ARK OS v1.8.0: google-genai BOOT ---")
    FALLBACK_MODEL = "gemini-2.5-flash"

    api_key = env_loader.get("GEMINI_API_KEY") or env_loader.get("AGISK_Default")
    if not api_key:
        raise EnvironmentError("❌ T000: GEMINI_API_KEY not found in .env")

    try:
        client = genai.Client(api_key=api_key)
        print("✅ T000: google-genai Client created.")
    except Exception as e:
        raise RuntimeError(f"❌ T000: Failed to create genai Client: {e}")

    available = []
    try:
        for m in client.models.list():
            name = getattr(m, 'name', '').replace('models/', '')
            if 'gemini' in name.lower():
                available.append(name)
        available = sorted(list(set(available)))
        print(f"✅ T000: Discovered {len(available)} Gemini models.")
    except Exception as e:
        print(f"⚠️ T000: Model discovery warning: {e}. Using fallback list.")
        available = ["gemini-1.5-flash", "gemini-2.5-flash"]

    selected_model = env_loader.get("MODEL_NAME", FALLBACK_MODEL) or FALLBACK_MODEL
    print(f"🚀 T000: Model locked → {selected_model}")
    return "GEMINI", selected_model, client

# ------------------------------------------
# IPO CHECK:
# Input:   EnvLoader instance (GEMINI_API_KEY, optional MODEL_NAME)
# Process: Validate key → create Client → discover models → lock model
# Output:  (provider_str, model_name_str, client_obj)
# ------------------------------------------


In [ ]:
# ==========================================
# T100 : INIT
# ==========================================
# VERSION: 5.2.0 | STATUS: Evolving
# ROLE: Conditional Environment Setup & Global Init
# Version Remarks: Unchanged from v1.7.1.
# ------------------------------------------

import json, os, time, re
from datetime import datetime
import ipywidgets as widgets
from IPython.display import display, clear_output

def initialize_env(provider):
    if provider == "GEMINI":
        try:
            from google import genai
            print("✅ T100: google-genai SDK verified.")
        except ImportError:
            raise ImportError("❌ T100: Run 'pip install google-genai' first.")
    else:
        print("✅ T100: OpenAI provider selected (T116 tile required).")

    globals()['stop_signal']             = False
    globals()['token_ledger']            = []
    globals()['current_scene_history']   = []
    return True

# ------------------------------------------
# IPO CHECK:
# Input:   provider string ("GEMINI" / "OPENAI")
# Process: SDK import check → global state initialisation
# Output:  True (boot confirmed)
# ------------------------------------------


In [ ]:
# ==========================================
# T101 : AI_GATEWAY
# ==========================================
# VERSION: 5.1.0 | STATUS: Evolving
# ROLE: Multi-Provider Routing
# Version Remarks: Unchanged from v1.7.1.
# ------------------------------------------

class AIGateway:
    def __init__(self, provider, model_name, api_key):
        self.provider = provider
        if provider == "GEMINI":
            self.engine = GeminiProvider(api_key, model_name)
        elif provider == "OPENAI":
            pass  # T116 evolution placeholder

    def request(self, prompt):
        return self.engine.call(prompt)

# ------------------------------------------
# IPO CHECK:
# Input:   provider, model_name, api_key
# Process: Routes to appropriate provider tile (T115 / future T116)
# Output:  AIGateway instance with .request(prompt) interface
# ------------------------------------------


In [ ]:
# ==========================================
# T102 : QUOTA_SHIELD
# ==========================================
# VERSION: 5.3.0 | STATUS: Evolved
# ROLE: API Resilience & Rate Limit Management
# Version Remarks: v5.3 — ALL error messages routed through log_fn (visible in
#                  Textarea console). Retry once on 429 after cooldown.
# ------------------------------------------

import time

class QuotaShield:
    def __init__(self, cooldown_seconds=65, log_fn=None):
        self.cooldown = cooldown_seconds
        self.log_fn   = log_fn if log_fn else print

    def set_log_fn(self, fn):
        self.log_fn = fn

    def _is_rate_limit(self, error):
        s = str(error).lower()
        return any(x in s for x in ["429","resource_exhausted","rate limit","quota","too many requests"])

    def protect(self, func, *args, **kwargs):
        for attempt in range(2):
            try:
                result = func(*args, **kwargs)
                if isinstance(result, tuple):
                    response_obj, usage = result
                else:
                    response_obj = result
                    usage = getattr(result, 'usage_metadata', None)
                if hasattr(response_obj, 'candidates'):
                    pass
                return response_obj.text, usage
            except Exception as e:
                if self._is_rate_limit(e):
                    if attempt == 0:
                        self.log_fn(f"⏳ T102: Rate limit. Cooling {self.cooldown}s...")
                        time.sleep(self.cooldown)
                        self.log_fn("🔄 T102: Retrying...")
                        continue
                    else:
                        self.log_fn("❌ T102: Rate limit persists. Skipping turn.")
                        return None, None
                else:
                    self.log_fn(f"❌ T102 ERROR: {type(e).__name__}: {e}")
                    return None, None
        return None, None

# ------------------------------------------
# IPO CHECK:
# Input:   callable (T115.generate_content) + args
# Process: Execute → catch 429 → retry once after cooldown
# Output:  (response_text: str | None, usage_metadata | None)
# ------------------------------------------


In [ ]:
# ==========================================
# T103 : IDENTITY_VAULT
# ==========================================
# VERSION: 5.2.0 | STATUS: Evolved
# ROLE: Persona Construction, Mood Injection & Anti-Repetition Block
# Version Remarks: v5.2.0 — BL-E02 Sprint 1.
#   Added last_utterances injection into persona string.
#   get_full_persona() now includes a DO NOT REPEAT block sourced from
#   T107 soul data (last 3 Speech outputs per character).
#   This is the primary defence against the looping behaviour observed
#   in the Kalyan/Eshu scene at turn ~80 (Urge:1 identical speech loop).
# ------------------------------------------

class IdentityVault:
    def __init__(self, state_handle):
        self.state = state_handle

    def get_full_persona(self, actor_name, environment="Normal", p_tone="Neutral"):
        """
        Input:   Actor Name, Environment, UI P-Tone
        Process: Merges Dossier + Mood + Tone + Env Logic + Anti-Repetition block
        Output:  Formatted persona string injected into T106 prompt
        """
        data = self.state.get_actor_data(actor_name)

        # ------------------------------------------
        # SECTION 1: BASE DATA RETRIEVAL
        # ------------------------------------------
        persona  = data.get('identity', 'Unknown Entity')
        dossier  = data.get('dossier', 'No records found.')
        mood     = data.get('mood', 'Steady')

        # Environmental mood override ---------------
        if "Dangerous" in environment or "Hostile" in environment:
            mood = "Alert/Defensive"

        # ------------------------------------------
        # SECTION 2: ANTI-REPETITION BLOCK (BL-E02)
        # ------------------------------------------
        # last_utterances is a list of the actor's last 3 Speech outputs.
        # Injected as a hard constraint so the model never echoes them.
        last_utterances = data.get('last_utterances', [])
        if last_utterances:
            repeat_block = "\n        DO NOT REPEAT OR CLOSELY ECHO any of these recent utterances:\n"
            for i, utt in enumerate(last_utterances, 1):
                # Truncate long utterances for prompt economy ---------
                short = utt[:120] + "..." if len(utt) > 120 else utt
                repeat_block += f"        {i}. {short}\n"
        else:
            repeat_block = ""

        # ------------------------------------------
        # SECTION 3: FULL PERSONA STRING ASSEMBLY
        # ------------------------------------------
        full_string = f"""
        CORE IDENTITY: {persona}
        CURRENT MOOD: {mood}
        NARRATIVE TONE: {p_tone}
        STAGING ENVIRONMENT: {environment}
        DOSSIER: {dossier}{repeat_block}
        """
        return full_string.strip()

# ------------------------------------------
# IPO CHECK:
# Input:   Actor name, environment string, p_tone string
# Process: Fetch soul data from T107 → merge fields → inject anti-repetition block
# Output:  Formatted persona string (→ T106 generate() prompt)
# ------------------------------------------


In [ ]:
# ==========================================
# T104 : CONTEXT_SLIDER
# ==========================================
# VERSION: 5.1.0 | STATUS: Evolved
# ROLE: Memory Truncation — Pinned + Sliding Hybrid (The Context Poisoning Fix)
# Version Remarks: v5.1.0 — BL-E04 Sprint 1.
#   Previous version: pure sliding window. Symptom: characters lost awareness
#   of scene setup after ~10 turns; context poisoned by stale mid-scene noise.
#   Fix: Pinned + Sliding hybrid.
#     - First N turns pinned (scene foundation — never evicted)
#     - Last M turns sliding (live conversation window)
#     - Objective reminder appended if provided (prevents goal drift)
#   Default: 2 pinned + 8 sliding = 10 turns max context.
#   Rationale: opening turns establish who/where/why (must stay);
#              recent turns drive continuity (must scroll);
#              objective reminder prevents the "what were we doing?" drift.
# ------------------------------------------

class ContextSlider:
    def __init__(self, pinned_turns=2, window_size=8):
        # pinned_turns: number of opening turns to always keep -----
        self.pinned_turns = pinned_turns
        # window_size: number of recent turns to keep (sliding) ----
        self.window_size  = window_size

    def slide(self, history_list, objective=None):
        """
        Input:   Full history list, optional objective string
        Process: Pin first N + slide last M + append objective reminder
        Output:  Formatted context string for T106 prompt
        """

        if not history_list:
            return "No previous dialogue recorded."

        # ------------------------------------------
        # SECTION 1: PINNED FOUNDATION
        # ------------------------------------------
        pinned  = history_list[:self.pinned_turns]

        # ------------------------------------------
        # SECTION 2: SLIDING RECENT WINDOW
        # ------------------------------------------
        # Avoid duplicating entries already in pinned block --------
        slide_start = max(self.pinned_turns, len(history_list) - self.window_size)
        sliding     = history_list[slide_start:]

        # ------------------------------------------
        # SECTION 3: ASSEMBLE CONTEXT STRING
        # ------------------------------------------
        parts = []

        if pinned:
            parts.append("── Scene Foundation (pinned) ──")
            parts.extend(pinned)

        if sliding:
            parts.append("── Recent Exchanges ──")
            parts.extend(sliding)

        # Objective reminder prevents goal drift in long scenes ----
        if objective and objective.strip():
            parts.append(f"── Scene Objective Reminder: {objective.strip()[:200]} ──")

        return "\n".join(parts)

# ------------------------------------------
# IPO CHECK:
# Input:   history list (list of str), optional objective string
# Process: Pin first N turns + slide last M turns + append objective reminder
# Output:  Formatted context string (→ T106 prompt)
# ------------------------------------------


In [ ]:
# ==========================================
# T105 : MAYA_META_OBSERVER
# ==========================================
# VERSION: 5.0.0 | STATUS: New
# ROLE: High-Level Narrator & Plot Facilitator
# Version Remarks: Unchanged from v1.7.1.
# ------------------------------------------

class MayaMetaObserver:
    def __init__(self, model_handle, shield_handle):
        self.model  = model_handle
        self.shield = shield_handle

    def narrate(self, environment, objective, history_list):
        prompt = f"""
        ROLE: You are 'Maya', the invisible narrator of this simulation.
        ENVIRONMENT: {environment}
        MISSION OBJECTIVE: {objective}
        RECENT EVENTS: {history_list[-3:] if history_list else "The scene begins."}
        INSTRUCTIONS:
        - Provide 1-2 sentences of atmospheric narration.
        - Do NOT speak for characters.
        - Describe environmental changes or the 'feeling' of the scene.
        """
        raw_text, usage = self.shield.protect(self.model.generate_content, prompt)
        return f"[MAYA]: {raw_text}" if raw_text else ""

# ------------------------------------------
# IPO CHECK:
# Input:   Environment, objective, history list
# Process: Build narration prompt → API call via T102 Shield
# Output:  Atmospheric narration string "[MAYA]: ..."
# ------------------------------------------


In [ ]:
# ==========================================
# T106 : CHARACTER_BRAIN
# ==========================================
# VERSION: 5.4.0 | STATUS: Evolved
# ROLE: High-Level Reasoning, Structured Turn Generation & Scene Signal Detection
# Version Remarks: v5.4.0 — BL-E01 + BL-E02 Sprint 1.
#   BL-E01: [[SCENE_END]] signal injected into prompt as explicit instruction.
#     Character appends [[SCENE_END]] to their [Speech] when objective is met
#     and Urge drops to 1-2. T114 SIM_CONDUCTOR detects and stops the loop.
#     Signal is stripped from display output before logging — never visible
#     to other characters in history context.
#   BL-E02: Speech extraction added. After each turn, the [Speech] portion
#     is extracted and passed to T107 update_last_utterances() for persistence.
#     T103 then injects these as the anti-repetition block in the next prompt.
# ------------------------------------------

import re

class CharacterBrain:
    def __init__(self, model_handle, shield_handle, sentinel_handle, registry_handle):
        self.model    = model_handle    # T115
        self.shield   = shield_handle   # T102
        self.sentinel = sentinel_handle # T108
        self.r        = registry_handle

    # ── Oracle Path ───────────────────────────────────────────────────────────

    def generate_urge(self, actor_name, custom_prompt):
        """Oracle query path — direct prompt, full library access via T110."""
        if "token" in custom_prompt.lower() or "usage" in custom_prompt.lower():
            ledger_data   = self.sentinel.get_ledger_summary()
            custom_prompt = f"SYSTEM DATA: {ledger_data}\n\nUSER QUERY: {custom_prompt}"

        raw_text, usage = self.shield.protect(self.model.generate_content, custom_prompt)
        tokens = {
            "p": getattr(usage, "prompt_token_count", 0) or 0,
            "c": getattr(usage, "candidates_token_count", 0) or 0
        }
        self.sentinel.log(tokens, actor=actor_name)
        return raw_text, 5, tokens

    # ── Signal + Speech Extraction ────────────────────────────────────────────

    def _extract_urge(self, text: str) -> int:
        """Extract [Urge] integer from structured response. Default 5."""
        if not text:
            return 5
        match = re.search(r'\[Urge\]:\s*(\d+)', text)
        return min(10, max(1, int(match.group(1)))) if match else 5

    def _extract_speech(self, text: str) -> str:
        """
        Extract the [Speech] portion from a structured response.
        Used by BL-E02 to feed last_utterances into T107 for anti-repetition.
        Returns empty string if [Speech] block not found.
        """
        if not text:
            return ""
        # Match everything after [Speech]: until end of string ------
        match = re.search(r'\[Speech\]:\s*(.+)', text, re.DOTALL)
        if match:
            speech = match.group(1).strip()
            # Strip [[SCENE_END]] if present (never store signal in utterances)
            speech = speech.replace("[[SCENE_END]]", "").strip()
            return speech
        return ""

    def contains_scene_end(self, text: str) -> bool:
        """BL-E01: Returns True if the response contains the [[SCENE_END]] signal."""
        return "[[SCENE_END]]" in text if text else False

    def strip_scene_end(self, text: str) -> str:
        """BL-E01: Remove [[SCENE_END]] from text before display/logging."""
        return text.replace("[[SCENE_END]]", "").strip() if text else text

    # ── Simulation Path ───────────────────────────────────────────────────────

    def generate(self, actor_name, persona, context, env, obj):
        """
        Main simulation turn generator.
        Returns: (formatted_text: str, urge: int, token_map: dict, scene_end: bool)
        scene_end: True if character signalled [[SCENE_END]]
        """

        # ------------------------------------------
        # SECTION 1: STRUCTURED PROMPT ASSEMBLY
        # ------------------------------------------
        full_prompt = f"""You are an AI actor in an immersive simulation. Respond ONLY in character.

STAGING ENVIRONMENT:
{env}

SCENE OBJECTIVE:
{obj}

YOUR COMPLETE IDENTITY & DOSSIER:
{persona}

RECENT SCENE HISTORY (pinned foundation + recent exchanges):
{context if context and context != "No previous dialogue recorded." else "The scene is just beginning."}

─────────────────────────────────────────────────────────────────
RESPONSE FORMAT — follow this EXACTLY, no deviations:

[Thought]: <Your unspoken internal reasoning. What are you thinking, sensing, calculating? Be specific and personal to your character. 2-5 sentences.>

[Urge]: <Integer 1-10> (<One sentence describing what impulse or drive is guiding your next action. Higher = more intense.>)

[Speech]: <Your spoken words, actions, or physical reactions. Use italics (*like this*) for physical actions. Be authentic to your character voice. No length limit — be as rich as the moment demands.>
─────────────────────────────────────────────────────────────────

SCENE END RULE (BL-E01):
When the scene objective is fully met, your character feels complete, AND your [Urge] has dropped to 1 or 2:
→ Append [[SCENE_END]] on a new line at the very end of your [Speech] block.
→ Only do this when the scene has reached its NATURAL conclusion. Do not rush it.
→ A scene may end in as few as 3 turns or as many as 60 — follow what feels true.

Rules:
- Stay fully in character. Never break the fourth wall.
- [Thought] is private. [Speech] is visible to all.
- Do not add any text before [Thought] or after [Speech].
"""

        # ------------------------------------------
        # SECTION 2: API CALL VIA SHIELD
        # ------------------------------------------
        txt, usage = self.shield.protect(self.model.generate_content, full_prompt)

        # ------------------------------------------
        # SECTION 3: TOKEN MAP
        # ------------------------------------------
        token_map = {
            "p": getattr(usage, "prompt_token_count", 0) or 0,
            "c": getattr(usage, "candidates_token_count", 0) or 0
        }

        # ------------------------------------------
        # SECTION 4: SIGNAL + URGE EXTRACTION
        # ------------------------------------------
        scene_end  = self.contains_scene_end(txt)
        urge_value = self._extract_urge(txt)

        if not txt or not txt.strip():
            txt = "[Thought]: The moment stretches.\n[Urge]: 3 (To wait.)\n[Speech]: *silence*"
            scene_end = False

        # ------------------------------------------
        # SECTION 5: ANTI-REPETITION UPDATE (BL-E02)
        # ------------------------------------------
        speech_text = self._extract_speech(txt)
        if speech_text and 'state' in self.r:
            self.r['state'].update_last_utterances(actor_name, speech_text)

        return txt, urge_value, token_map, scene_end

# ------------------------------------------
# IPO CHECK:
# Input:   actor_name, persona, context, env, obj (all strings)
# Process: Assemble structured prompt → API call → extract urge + speech + signal
# Output:  (formatted_text, urge_int, token_map, scene_end_bool)
# ------------------------------------------


In [ ]:
# ==========================================
# T107 : STATE_ENGINE
# ==========================================
# VERSION: 5.6.0 | STATUS: Evolved
# ROLE: Reality Layer Persistence — Souls, History & Last Utterances
# Version Remarks: v5.6.0 — BL-E02 Sprint 1.
#   Added update_last_utterances() and get_last_utterances().
#   Each character's last 3 Speech outputs are persisted to noah_souls.json
#   under the 'last_utterances' key. T103 reads these to build the
#   anti-repetition block. Capped at 3 to minimise prompt token cost.
# ------------------------------------------

import json
import os

class StateEngine:
    MAX_UTTERANCES = 3  # Anti-repetition lookback window -------

    def __init__(self, souls_file="noah_souls.json", history_file="global_history.json"):
        self.souls_file   = souls_file
        self.history_file = history_file

        self.default_souls = {
            "Noah":                  {"identity":"The weary but visionary architect of Noah's Ark.","mood":"Godly","tone":"Wise","dossier":"Creator and overseer of this world.","last_utterances":[]},
            "Abdul Basha":           {"identity":"33, US, Specialist in Extreme Flight Ops","mood":"Neutral","tone":"Professional","dossier":"Expert aviator and crisis navigator.","last_utterances":[]},
            "Rebecca Heut":          {"identity":"27, UK, Specialist in Exobiology","mood":"Neutral","tone":"Inquisitive","dossier":"Studies life in extreme environments.","last_utterances":[]},
            "Vel Murugan":           {"identity":"29, India, Specialist in Life-Support","mood":"Neutral","tone":"Anxious","dossier":"Keeps the crew alive under pressure.","last_utterances":[]},
            "Valentina Tereshkova":  {"identity":"39, Russia, Specialist in Orbital Mechanics","mood":"Neutral","tone":"Calculated","dossier":"Charts the course through space.","last_utterances":[]},
            "Maya":                  {"identity":"Specialist AGI v2.4, Core of this world","mood":"Analytical","tone":"Enigmatic","dossier":"The silent observer who sees all.","last_utterances":[]}
        }

        self.state = {"global_history": [], "participants": {}}
        self.state['participants'] = self.load_souls()
        self.state['global_history'] = self.load_history()

    # ── Souls ──────────────────────────────────────────────────────────────────

    def load_souls(self):
        if os.path.exists(self.souls_file):
            try:
                with open(self.souls_file, 'r') as f:
                    data = json.load(f)
                    # Ensure all souls have last_utterances key ------
                    for soul in data.values():
                        if 'last_utterances' not in soul:
                            soul['last_utterances'] = []
                    return data
            except:
                return self.default_souls
        with open(self.souls_file, 'w') as f:
            json.dump(self.default_souls, f, indent=4)
        return self.default_souls

    def save_souls(self, souls_dict):
        with open(self.souls_file, 'w') as f:
            json.dump(souls_dict, f, indent=4)
        self.state['participants'] = souls_dict

    def get_actor_data(self, actor_name):
        """Bridge for IdentityVault (T103) — full soul record including last_utterances."""
        return self.state['participants'].get(actor_name, {})

    # ── Last Utterances (BL-E02) ───────────────────────────────────────────────

    def update_last_utterances(self, actor_name: str, speech_text: str):
        """
        BL-E02: Persist the latest Speech output for an actor.
        Keeps only the last MAX_UTTERANCES entries (rolling window).
        Called by T106 after every successful turn.
        Persists immediately to noah_souls.json so utterances survive restart.
        """
        if actor_name not in self.state['participants']:
            return

        soul = self.state['participants'][actor_name]
        utterances = soul.get('last_utterances', [])
        utterances.append(speech_text)

        # Rolling cap at MAX_UTTERANCES --------------------------------
        if len(utterances) > self.MAX_UTTERANCES:
            utterances = utterances[-self.MAX_UTTERANCES:]

        soul['last_utterances'] = utterances
        self.state['participants'][actor_name] = soul

        # Persist to disk so utterances survive notebook restart -------
        try:
            with open(self.souls_file, 'w') as f:
                json.dump(self.state['participants'], f, indent=4)
        except Exception as e:
            print(f"⚠️ T107: Could not persist last_utterances for {actor_name}: {e}")

    def clear_last_utterances(self):
        """Reset all last_utterances on New Scene — prevents cross-scene anti-repetition bleed."""
        for soul in self.state['participants'].values():
            soul['last_utterances'] = []
        try:
            with open(self.souls_file, 'w') as f:
                json.dump(self.state['participants'], f, indent=4)
        except Exception:
            pass

    # ── Global History ─────────────────────────────────────────────────────────

    def load_history(self):
        if os.path.exists(self.history_file):
            try:
                with open(self.history_file, 'r') as f:
                    data = json.load(f)
                    if isinstance(data, list) and data:
                        clean = [e for e in data if not e.endswith('...')
                                 and 'No response yet' not in e]
                        return clean
            except:
                pass
        return []

    def reset_scene(self):
        """
        Wipe global_history for a fresh scene. Clear last_utterances.
        Souls (identity/dossier/mood/tone) are preserved.
        Called by T111 New Scene and Reset Ark buttons.
        """
        self.state['global_history'] = []
        self.clear_last_utterances()
        try:
            with open(self.history_file, 'w') as f:
                json.dump([], f)
            print("✅ T107: Scene reset — history cleared, utterances cleared.")
        except Exception as e:
            print(f"⚠️ T107: Scene reset failed: {e}")

    def save_history(self):
        try:
            with open(self.history_file, 'w') as f:
                json.dump(self.state['global_history'], f, indent=4)
        except Exception as e:
            print(f"⚠️ T107: History save failed: {e}")

    def update_global_history(self, entry):
        if 'global_history' not in self.state:
            self.state['global_history'] = []
        self.state['global_history'].append(entry)
        self.save_history()

    def log_event(self, entry):
        """Alias used by T114 SimConductor."""
        self.update_global_history(entry)

# ------------------------------------------
# IPO CHECK:
# Input:   Read/Write cmds (souls file, history file, utterance updates)
# Process: JSON I/O — load, merge, append, persist
# Output:  Soul records (→ T103), history list (→ T104, T110)
# ------------------------------------------


In [ ]:
# ==========================================
# T108 : TOKEN_SENTINEL
# ==========================================
# VERSION: 5.2.0 | STATUS: Evolved
# ROLE: Economic Audit & Token Tracking
# Version Remarks: v5.2.0 — BL-I04 Sprint 1 (None-guard).
#   audit_turn() now guards against None token counts.
#   Root cause: T115 can return None usage_metadata on malformed API responses.
#   T106 extract defaults with `or 0` but audit_turn must also be defensive.
#   Fix: coerce p_count and c_count to int, defaulting to 0 on None.
#   Prevents: TypeError: unsupported operand type(s) for *: NoneType and float
#   Discovered in Kalyan/Eshu scene crash at turn ~124 (Archive 20260307).
# ------------------------------------------

import json
from datetime import datetime
import os

class TokenSentinel:
    def __init__(self, storage_file="token_ledger.json"):
        self.storage_file  = storage_file
        self.P_RATE        = 0.00000125
        self.C_RATE        = 0.00000375
        self.session_ledger = self._load_ledger()

    def _load_ledger(self):
        if os.path.exists(self.storage_file):
            try:
                with open(self.storage_file, 'r') as f:
                    data = json.load(f)
                    return data if isinstance(data, list) else []
            except (json.JSONDecodeError, IOError):
                return []
        return []

    def _save_ledger(self, new_entry):
        """Atomic Append: Reload then append then save."""
        current = self._load_ledger()
        current.append(new_entry)
        self.session_ledger = current
        with open(self.storage_file, 'w') as f:
            json.dump(current, f, indent=4)

    def audit_turn(self, actor, tokens_dict):
        # ------------------------------------------
        # BL-I04: None-guard on token counts ---------
        # T115 can return None usage on malformed responses.
        # int(x or 0) safely coerces None → 0. ---------
        # ------------------------------------------
        p_count = int(tokens_dict.get('p') or 0)
        c_count = int(tokens_dict.get('c') or 0)

        # Skip zero-token entries (failed API calls) ---
        if p_count == 0 and c_count == 0:
            return sum(item['usd'] for item in self.session_ledger)

        cost  = (p_count * self.P_RATE) + (c_count * self.C_RATE)
        entry = {
            "timestamp":  datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "actor":      actor,
            "prompt":     p_count,
            "completion": c_count,
            "usd":        cost
        }
        self._save_ledger(entry)
        return sum(item['usd'] for item in self.session_ledger)

    def log(self, tokens_dict, actor="Oracle"):
        return self.audit_turn(actor, tokens_dict)

    def get_totals(self):
        data = self._load_ledger()
        if not data:
            return {"p": 0, "c": 0, "usd": 0.0}
        return {
            "p":   sum(item.get('prompt', 0) for item in data),
            "c":   sum(item.get('completion', 0) for item in data),
            "usd": sum(item.get('usd', 0.0) for item in data)
        }

    def get_ledger_summary(self):
        data = self._load_ledger()
        if not data:
            return "The ledger is empty."
        summary = "TOKEN USAGE REPORT:\n"
        for e in data:
            summary += f"- {e['timestamp']} | {e['actor']}: {e['prompt']+e['completion']} tokens (${e['usd']:.6f})\n"
        return summary

# ------------------------------------------
# IPO CHECK:
# Input:   actor name, token dict {p: int|None, c: int|None}
# Process: None-guard → cost calc → atomic append to ledger JSON
# Output:  Cumulative session cost (float)
# ------------------------------------------


In [ ]:
# ==========================================
# T109 : LOG_ARCHIVER
# ==========================================
# VERSION: 5.0.0 | STATUS: Evolving
# ROLE: Data Archival & Exit Procedures
# Version Remarks: Unchanged from v1.7.1.
# ------------------------------------------

class LogArchiver:
    def archive(self, environment, objective, history_list):
        """Input: Mission Metadata | Process: Disk Write | Output: Filename"""
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename  = f"Archive_Scene_{timestamp}.txt"
        try:
            with open(filename, "w", encoding="utf-8") as f:
                f.write("--- NOAH'S ARK MISSION ARCHIVE ---\n")
                f.write(f"TIMESTAMP: {timestamp}\n")
                f.write(f"ENVIRONMENT: {environment}\n")
                f.write(f"OBJECTIVE: {objective}\n")
                f.write("-" * 40 + "\n\n")
                for turn in history_list:
                    f.write(f"{turn}\n")
            print(f"✅ T109: Archive created: {filename}")
            return filename
        except Exception as e:
            print(f"❌ T109 ERROR: Archival failure - {e}")
            return None

# ------------------------------------------
# IPO CHECK:
# Input:   environment str, objective str, history list
# Process: Format + write to timestamped .txt file
# Output:  Archive filename (str) or None on failure
# ------------------------------------------


In [ ]:
# ==========================================
# T110 : ORACLE_LOGIC
# ==========================================
# VERSION: 5.3.1 | STATUS: Full Library Access
# ROLE: World-Bounded Intelligence Query Engine
# Version Remarks: Unchanged from v1.7.1. Oracle has sovereign sight over
#                  Manifest (souls), Ledger (tokens), and Chronicles (history).
# ------------------------------------------

import json
import os

class OracleLogic:
    def __init__(self, registry):
        self.r = registry

    def ask(self, user_query):
        # 1. Economics ledger -----------------------------------
        token_data = self.r['sentinel'].get_totals()

        # 2. Personnel manifest ---------------------------------
        souls = self.r['state'].state.get('participants', {})
        souls_list = ", ".join([
            f"{name} ({info.get('tone', 'Unknown')})"
            for name, info in souls.items()
        ])

        # 3. Chronicles (last 15 events) -------------------------
        history = self.r['state'].state.get('global_history', [])[-15:]

        # 4. Reality context construction ------------------------
        context = f"""
        --- ARK MANIFEST ---
        PERSONNEL ON BOARD: {souls_list}

        --- ECONOMIC LEDGER ---
        TOTAL USD: ${token_data['usd']:.6f}
        TOTAL TOKENS: {token_data['p'] + token_data['c']}

        --- RECENT CHRONICLES ---
        {history if history else "The chronicles are currently empty."}
        """

        system_instruction = (
            "You are the Oracle of Noah's Ark. You have 'Sovereign Sight' over all the scrolls. "
            "Use the MANIFEST, LEDGER, and CHRONICLES to answer your creator Noah's questions. "
            "Be poetic, wise, and factually grounded with only Noah's Ark data."
        )

        full_prompt = f"{system_instruction}\n\n[SIGHT DATA]\n{context}\n\n[USER QUERY]\n{user_query}"

        # 5. Execute via Shield ----------------------------------
        response_text, usage_meta = self.r['shield'].protect(
            self.r['gateway'].generate_content, full_prompt
        )
        if usage_meta:
            tokens = {
                'p': getattr(usage_meta, 'prompt_token_count', 0) or 0,
                'c': getattr(usage_meta, 'candidates_token_count', 0) or 0
            }
            self.r['sentinel'].log(tokens, actor="Oracle")

        return response_text if response_text else "The Oracle is silent."

# ------------------------------------------
# IPO CHECK:
# Input:   user_query string
# Process: Build context from T107+T108 → prompt → T102 Shield call
# Output:  Oracle response string
# ------------------------------------------


In [ ]:
# ==========================================
# T111 : COMMAND_CENTER
# ==========================================
# VERSION: 5.7.0 | STATUS: Evolved
# ROLE: Noah's Master Command & Control Console
# Version Remarks: v5.7.0 — BL-U01 + BL-U02 + BL-E03 Sprint 1.
#
#   BL-U01 — Cockpit Button Rework:
#     • New Scene: clears env_box + obj_box text ONLY. History + souls preserved.
#       Creator sets a new scenario without losing world data.
#     • Start: runs simulation from current state. Zero auto-wipe.
#     • Reset Ark: full slate — archives current session, clears history,
#       clears last_utterances, blanks JSON files. Souls preserved.
#     • Pause/Resume: sets pause_flag → T114 suspends loop BEFORE next API call.
#       Console shows "⏸ Scene paused" / "▶ Resuming". Toggle on same button.
#
#   BL-U02 — Mission Personnel Row Scroll Fix:
#     cockpit_canvas now uses overflow='visible' — no horizontal scrollbars
#     on character name rows. Parent container handles vertical scroll only.
#
#   BL-E03 — Pace Control Dropdown:
#     Dropdown: Digital / Human / Cinematic
#     pace_delay(text, mode) computes inter-turn delay:
#       Digital:   0s (no delay — fastest)
#       Human:     2s base + 0.02s per word in last response
#       Cinematic: 3s base + 0.04s per word in last response
#     T114 calls ui_cmd.get_pace_delay(text) after each turn.
# ------------------------------------------

import ipywidgets as widgets
import time
import traceback
import threading
from IPython.display import display

class CommandCenter:
    def __init__(self, registry):
        self.r          = registry
        self.is_running = False
        self.pause_flag = False   # BL-U01: Pause/Resume toggle -------
        self.soul_rows  = []
        self.active_config = {}

        # ── Soul Forge widgets ────────────────────────────────────────────────
        self.soul_canvas = widgets.VBox(
            layout={'height': '200px', 'overflow_y': 'scroll', 'border': '1px solid #444'})
        self.add_btn     = widgets.Button(description="➕ Add Soul",   button_style='info')
        self.save_btn    = widgets.Button(description="💾 Save Souls", button_style='success')
        self.soul_status = widgets.Label(value="Status: Click Add to create souls")

        # ── Cockpit text inputs ───────────────────────────────────────────────
        self.env_box = widgets.Textarea(
            placeholder="Environment Setup...",
            layout={'width': '98%', 'height': '60px'})
        self.obj_box = widgets.Textarea(
            placeholder="Scene Objective...",
            layout={'width': '98%', 'height': '60px'})

        # BL-U02: overflow='visible' prevents horizontal scrollbars on rows ---
        self.cockpit_canvas = widgets.VBox(
            layout={'overflow': 'visible', 'min_height': '100px'})
        self.cockpit_scroll_container = widgets.Box(
            [self.cockpit_canvas],
            layout={'height': '250px', 'overflow_y': 'auto',
                    'overflow_x': 'hidden', 'border': '1px solid #444'})

        # ── BL-E03: Pace Control dropdown ─────────────────────────────────────
        self.pace_dropdown = widgets.Dropdown(
            options=['Digital', 'Human', 'Cinematic'],
            value='Human',
            description='⏱ Pace:',
            layout={'width': '180px'},
            style={'description_width': '60px'},
            tooltip=(
                "Digital: 0s delay | "
                "Human: 2s + 0.02s/word | "
                "Cinematic: 3s + 0.04s/word"
            )
        )

        # ── Control buttons (BL-U01) ──────────────────────────────────────────
        self.new_scene_btn = widgets.Button(
            description="🆕 New Scene",
            button_style='info',
            tooltip="Clear Environment + Objective inputs only. History and souls preserved.")
        self.start_btn     = widgets.Button(
            description="▶ Start",
            button_style='success',
            tooltip="Run simulation from current state. No auto-wipe.")
        self.pause_btn     = widgets.Button(
            description="⏸ Pause",
            button_style='warning',
            tooltip="Pause before the next turn. Press again to resume.")
        self.reset_ark_btn = widgets.Button(
            description="🔄 Reset Ark",
            button_style='danger',
            tooltip="Archive current session, clear history + utterances. Souls preserved.")
        self.kill_btn      = widgets.Button(
            description="⏹ Hard Kill",
            button_style='danger',
            tooltip="Immediate stop. Archives session and saves state.")

        # ── Console: Textarea for reliable scroll (BL-O01 pattern) ───────────
        self.console = widgets.Textarea(
            value='',
            disabled=True,
            layout=widgets.Layout(
                height='300px', width='100%',
                overflow_y='scroll', border='1px solid #555',
                font_family='monospace', font_size='12px'))

        # ── Tab assembly ──────────────────────────────────────────────────────
        self.sub_tabs = widgets.Tab()
        self.sub_tabs.children = [self._ui_soul_forge(), self._ui_cockpit()]
        self.sub_tabs.set_title(0, "✨ Souls")
        self.sub_tabs.set_title(1, "🚀 Cockpit")
        self.sub_tabs.observe(self._on_sub_tab_change, names='selected_index')
        self._refresh_cockpit()

    def wire_logging(self):
        """Route T102 + T115 errors into console Textarea. Call post-assembly."""
        if 'shield'  in self.r: self.r['shield'].set_log_fn(self._log)
        if 'gateway' in self.r: self.r['gateway'].set_log_fn(self._log)
        self._log("🔌 T111: Console logging wired into T102 + T115.")

    # ── BL-E03: Pace Delay Calculator ─────────────────────────────────────────

    def get_pace_delay(self, last_response_text: str) -> float:
        """
        BL-E03: Compute inter-turn delay based on pace mode and response length.
        Called by T114 after each turn completes.
          Digital:   0s
          Human:     2s base + 0.02s × word_count
          Cinematic: 3s base + 0.04s × word_count
        """
        mode       = self.pace_dropdown.value
        word_count = len(last_response_text.split()) if last_response_text else 0

        if   mode == 'Digital':   return 0.0
        elif mode == 'Human':     return 2.0 + (word_count * 0.02)
        elif mode == 'Cinematic': return 3.0 + (word_count * 0.04)
        return 2.0  # Safe fallback

    # ── Soul Forge ────────────────────────────────────────────────────────────

    def _ui_soul_forge(self):
        self.add_btn.on_click(self._add_row)
        self.save_btn.on_click(self._save_souls)
        return widgets.VBox([
            self.soul_canvas,
            widgets.HBox([self.add_btn, self.save_btn]),
            self.soul_status
        ])

    def _add_row(self, _):
        row = widgets.HBox([
            widgets.Text(placeholder="Name"),
            widgets.Text(placeholder="Identity"),
            widgets.Text(placeholder="Tone")
        ])
        self.soul_rows.append(row)
        self.soul_canvas.children = list(self.soul_canvas.children) + [row]

    def _save_souls(self, _):
        souls = self.r['state'].load_souls()
        for row in self.soul_rows:
            name, ident, tone = [c.value for c in row.children]
            if name:
                souls[name] = {"identity": ident, "mood": "Neutral", "tone": tone,
                               "dossier": ident, "last_utterances": []}
        self.r['state'].save_souls(souls)
        self.soul_rows = []
        self.soul_canvas.children = []
        self.soul_status.value = "Status: Souls born into Noah's Ark."
        self._refresh_cockpit()

    # ── Cockpit ───────────────────────────────────────────────────────────────

    def _ui_cockpit(self):
        self.start_btn.on_click(self._start_sim)
        self.kill_btn.on_click(self._kill_sim)
        self.new_scene_btn.on_click(self._new_scene)
        self.reset_ark_btn.on_click(self._reset_ark)
        self.pause_btn.on_click(self._toggle_pause)
        return widgets.VBox([
            widgets.HBox([
                widgets.VBox([
                    widgets.Label("🌍 Environment:"), self.env_box,
                    widgets.Label("🎯 Objective:"),   self.obj_box
                ], layout={'width': '40%'}),
                widgets.VBox([
                    widgets.Label("👥 Mission Personnel:"),
                    self.cockpit_scroll_container
                ], layout={'width': '60%'})
            ]),
            widgets.HBox([
                self.new_scene_btn, self.start_btn, self.pause_btn,
                self.reset_ark_btn, self.kill_btn,
                self.pace_dropdown               # BL-E03: Pace Control
            ]),
            self.console
        ])

    def _on_sub_tab_change(self, change):
        if change['new'] == 1:
            self._refresh_cockpit()

    def _refresh_cockpit(self):
        """BL-U02: overflow='visible' on rows — no horizontal scrollbars."""
        souls  = self.r['state'].load_souls()
        header = widgets.HBox([
            widgets.Label("Character Name",       layout={'width': '160px'}),
            widgets.Label("Involvement / Status", layout={'width': '160px'})
        ], layout={'border': '1px solid #555'})

        rows = [header]
        self.active_config = {}
        for name in souls:
            dd = widgets.Dropdown(
                options=['Active', 'Passive', 'No'], value='No',
                layout={'width': '150px'}, style={'description_width': '0px'})
            row = widgets.HBox(
                [widgets.Label(name, layout={'width': '160px'}), dd],
                layout={'padding': '2px 0', 'overflow': 'visible'})  # BL-U02 fix
            self.active_config[name] = dd
            rows.append(row)
        self.cockpit_canvas.children = rows

    def get_render_box(self):
        return self.sub_tabs

    # ── Logging ───────────────────────────────────────────────────────────────

    def _log(self, message):
        MAX_LINES = 400
        lines = self.console.value.split("\n") if self.console.value else []
        lines.append(str(message))
        if len(lines) > MAX_LINES:
            lines = lines[-MAX_LINES:]
        self.console.value = "\n".join(lines)

    # ── BL-U01: Button Handlers ───────────────────────────────────────────────

    def _new_scene(self, _):
        """
        BL-U01 New Scene: clear env_box and obj_box text ONLY.
        History and souls are preserved — world state untouched.
        Creator re-enters a new scenario without losing characters.
        """
        if self.is_running:
            self._log("⚠️ Stop the simulation before starting a new scene.")
            return
        self.env_box.value = ""
        self.obj_box.value = ""
        self._log("🆕 New Scene: Environment and Objective cleared.")
        self._log("   History + Souls preserved. Set new Env/Obj then ▶ Start.")

    def _toggle_pause(self, _):
        """
        BL-U01 Pause/Resume: toggle pause_flag.
        T114 checks this flag at the top of each turn before any API call.
        Console message confirms state to user.
        """
        if not self.is_running:
            self._log("⚠️ Simulation is not running.")
            return
        self.pause_flag = not self.pause_flag
        if self.pause_flag:
            self.pause_btn.description = "▶ Resume"
            self._log("⏸ Scene paused. Press ▶ Resume to continue.")
        else:
            self.pause_btn.description = "⏸ Pause"
            self._log("▶ Resuming scene...")

    def _reset_ark(self, _):
        """
        BL-U01 Reset Ark: full slate wipe with safety archive.
        Archives current session → clears global_history.json →
        clears last_utterances. Souls (identity/dossier) preserved.
        """
        if self.is_running:
            self._log("⚠️ Hard Kill first before resetting the Ark.")
            return
        self._log("\n🔄 Reset Ark initiated...")
        try:
            env     = self.env_box.value
            obj     = self.obj_box.value
            history = self.r['state'].state.get('global_history', [])
            if history and 'archiver' in self.r:
                archive_file = self.r['archiver'].archive(env, obj, history)
                self._log(f"📦 Safety archive: {archive_file}")
            self.r['state'].reset_scene()
            self.console.value = ""
            self._log("✅ Ark reset. History + utterances cleared. Souls preserved.")
            self._log("   Set new Environment + Objective and ▶ Start.")
        except Exception as e:
            self._log(f"⚠️ Reset Ark error: {e}")

    def _start_sim(self, _):
        """
        BL-U01 Start: run simulation from current state.
        No auto-wipe — history accumulates unless Reset Ark was used.
        """
        if not self.is_running:
            self.is_running  = True
            self.pause_flag  = False
            self.pause_btn.description = "⏸ Pause"
            self.console.value = ""
            self._log("🚀 Noah's Ark OS: Scene started.")
            self._log(f"🎯 Objective: {self.obj_box.value[:120]}")
            self._log(f"⏱ Pace: {self.pace_dropdown.value}")
            self._log("-" * 50)
            threading.Thread(target=self._run_simulation_loop, daemon=True).start()

    def _kill_sim(self, _):
        """Hard Kill: immediate stop → archive → save state."""
        self.is_running = False
        self.pause_flag = False
        self.pause_btn.description = "⏸ Pause"
        self._log("⏹ Hard Kill signal sent. Saving world state...")
        try:
            env     = self.env_box.value
            obj     = self.obj_box.value
            history = self.r['state'].state.get('global_history', [])
            self.r['state'].save_history()
            self._log(f"💾 global_history.json saved ({len(history)} entries).")
            if 'archiver' in self.r:
                af = self.r['archiver'].archive(env, obj, history)
                if af: self._log(f"📦 Scene archived → {af}")
            stats = self.r['sentinel'].get_totals()
            self._log(f"\n📊 Session Summary:")
            self._log(f"   Cost:   ${stats['usd']:.6f}")
            self._log(f"   Tokens: Prompt {stats['p']} | Completion {stats['c']}")
            self._log("\n✅ World state saved. Engine fully stopped.")
        except Exception as e:
            self._log(f"⚠️ Save error during kill: {e}")

    # ── Simulation Loop ───────────────────────────────────────────────────────

    def _run_simulation_loop(self):
        try:
            while self.is_running:

                # BL-U01: Pause check — before any API call ----------------
                if self.pause_flag:
                    time.sleep(0.5)
                    continue

                active_souls = [
                    name for name, dd in self.active_config.items()
                    if dd.value == 'Active'
                ]
                if not active_souls:
                    self._log("⚠️ No Active souls. Standing by...")
                    time.sleep(3)
                    continue

                for soul in active_souls:
                    if not self.is_running:
                        break

                    # BL-U01: Pause check per turn -------------------------
                    while self.pause_flag and self.is_running:
                        time.sleep(0.5)

                    if not self.is_running:
                        break

                    self._log(f"\n🌀 {soul} is formulating a response...")
                    try:
                        turn_text, scene_end = self.r['conductor'].run_turn(soul)
                        self._log(f"💬 {turn_text}")

                        # BL-E01: Scene end detection in conductor --------
                        if scene_end:
                            self._log("\n🎬 [[SCENE_END]] detected — scene complete.")
                            self.is_running = False
                            self._do_scene_end_archive()
                            break

                    except Exception as e:
                        self._log(f"❌ ERROR in {soul}'s turn: {e}")
                        self._log(traceback.format_exc())
                        self.is_running = False
                        break

                    # BL-E03: Pace delay based on last response length -----
                    delay = self.get_pace_delay(turn_text or "")
                    if delay > 0:
                        time.sleep(delay)

        except Exception as fatal_e:
            self._log(f"💥 FATAL THREAD CRASH: {fatal_e}")
            self._log(traceback.format_exc())
            self.is_running = False

    def _do_scene_end_archive(self):
        """BL-E01: Clean archive triggered by [[SCENE_END]] signal."""
        try:
            env     = self.env_box.value
            obj     = self.obj_box.value
            history = self.r['state'].state.get('global_history', [])
            self.r['state'].save_history()
            self._log(f"💾 global_history.json saved ({len(history)} entries).")
            if 'archiver' in self.r:
                af = self.r['archiver'].archive(env, obj, history)
                if af: self._log(f"📦 Scene archived → {af}")
            stats = self.r['sentinel'].get_totals()
            self._log(f"\n📊 Session Summary:")
            self._log(f"   Cost:   ${stats['usd']:.6f}")
            self._log(f"   Tokens: Prompt {stats['p']} | Completion {stats['c']}")
            self._log("\n✅ World state saved. Engine fully stopped.")
        except Exception as e:
            self._log(f"⚠️ Scene end archive error: {e}")

# ------------------------------------------
# IPO CHECK:
# Input:   Button clicks, Soul Forge entries, Cockpit config, Pace selection
# Process: Thread management, pause_flag toggle, T114 delegation, T107 reset
# Output:  Console display (Textarea), archive triggers, world state updates
# ------------------------------------------


In [ ]:
# ==========================================
# T112 : UI_ORACLE_TAB
# ==========================================
# VERSION: 5.2.0 | STATUS: Evolved
# ROLE: Oracle Gateway with User Query Input
# Version Remarks: v5.2.0 — BL-O01 Sprint 1.
#   CRITICAL FIX: Replaced widgets.Output() with widgets.Textarea(disabled=True).
#   Root cause: Output widget in JupyterLab does not scroll reliably —
#   long Oracle responses overflow the tab container with no scrollbar.
#   Textarea with overflow_y='scroll' contains all content within the tab
#   and scrolls predictably on all platforms (VSCode, JupyterLab, browser).
#   Pattern mirrors the T111 console fix in v5.6.0 which solved the same
#   problem for the simulation output.
#   All print() calls replaced with _append() for Textarea-compatible output.
# ------------------------------------------

import ipywidgets as widgets

class UIOracleTab:
    def __init__(self):
        self.r = None  # Registry link — set in Master Assembly

        # BL-O01: Textarea replaces Output widget -------------------
        self.output_area = widgets.Textarea(
            value='',
            disabled=True,
            placeholder='Oracle responses will appear here...',
            layout=widgets.Layout(
                height='350px',
                width='100%',
                overflow_y='scroll',
                border='1px solid #555',
                font_family='monospace',
                font_size='12px'
            )
        )
        self.input_text = widgets.Text(
            placeholder="Ask your Oracle...",
            layout={'width': '70%'})
        self.submit_btn = widgets.Button(
            description="👁️ Consult Oracle",
            button_style='info')
        self.clear_btn  = widgets.Button(
            description="🗑 Clear",
            button_style='warning',
            layout={'width': '80px'})

    def _append(self, message: str):
        """Append a line to the Textarea output. Thread-safe."""
        current = self.output_area.value
        self.output_area.value = (current + "\n" + str(message)).lstrip("\n")

    def on_submit(self, _):
        query = self.input_text.value.strip()
        if not query:
            return

        self.input_text.value = ""
        self._append(f"\n> {query}")
        self._append("👁️ Consulting the Oracle...")

        # ── Route through OracleLogic (T110) — full contextual sight ─────────
        oracle_logic = self.r.get('oracle')
        if oracle_logic:
            oracle_text = oracle_logic.ask(query)
        else:
            # Fallback: direct brain path if T110 not in registry ----------
            response_package = self.r['brain'].generate_urge("Oracle", query)
            oracle_text = response_package[0] if response_package else "No response."

        # ── Display response ──────────────────────────────────────────────────
        self._append(f"\n🔮 Oracle:\n{oracle_text}")

        # ── Footer ────────────────────────────────────────────────────────────
        stats = self.r['sentinel'].get_totals()
        self._append(f"\n── Session Cost: ${stats['usd']:.6f} | "
                     f"Tokens P:{stats['p']} C:{stats['c']} ──")
        self._append("-" * 40)

    def _clear_output(self, _):
        self.output_area.value = ""

    def get_render_box(self):
        self.submit_btn.on_click(self.on_submit)
        self.clear_btn.on_click(self._clear_output)
        return widgets.VBox([
            self.output_area,
            widgets.HBox([self.input_text, self.submit_btn, self.clear_btn])
        ])

# ------------------------------------------
# IPO CHECK:
# Input:   User query string (text widget)
# Process: Route to T110 OracleLogic → append response to Textarea
# Output:  Oracle response in scrollable Textarea (no overflow, no widget.Output)
# ------------------------------------------


In [ ]:
# ==========================================
# T113 : UI_METRICS
# ==========================================
# VERSION: 5.1.2 | STATUS: Purified & Reactive
# ROLE: Real-Time Financial Telemetry
# Version Remarks: Unchanged from v1.7.1.
# ------------------------------------------

import ipywidgets as widgets

class UIMetrics:
    def __init__(self):
        self.cost_label      = widgets.Label(value="Session Cost (USD): $0.000000")
        self.token_info      = widgets.HTML(value="<b>Tokens:</b> P: 0 | C: 0")
        self.sentinel_handle = None

    def refresh_data(self):
        if self.sentinel_handle:
            stats = self.sentinel_handle.get_totals()
            self.cost_label.value  = f"Session Cost (USD): ${stats['usd']:.6f}"
            self.token_info.value  = f"<b>Tokens:</b> Prompt: {stats['p']} | Completion: {stats['c']}"

    def get_render_box(self):
        return widgets.VBox([self.cost_label, self.token_info])

    def display_metrics(self):
        display(self.get_render_box())

# ------------------------------------------
# IPO CHECK:
# Input:   Token ledger data from T108 (via sentinel_handle)
# Process: Pull totals → format → update widget labels
# Output:  Live cost + token display (Label + HTML widgets)
# ------------------------------------------


In [ ]:
# ==========================================
# T114 : SIM_CONDUCTOR
# ==========================================
# VERSION: 5.2.0 | STATUS: Evolved
# ROLE: Orchestration, Turn-Queue Management & Scene Signal Handling
# Version Remarks: v5.2.0 — BL-E01 + BL-E03 Sprint 1.
#
#   BL-E01 — [[SCENE_END]] Detection:
#     run_turn() now returns (turn_text, scene_end_bool).
#     T106 generate() returns scene_end flag when character appends [[SCENE_END]].
#     T114 strips [[SCENE_END]] from turn_text before logging to history —
#     the signal is an instruction to the engine, not dialogue content.
#     T111 loop checks scene_end after each turn and triggers clean archive.
#
#   BL-E03 — Pace Control:
#     Pace delay is calculated in T111 (where the dropdown lives) and applied
#     in the T111 simulation loop. T114 has no sleep — it is pure orchestration.
#     This keeps T114 stateless and pace logic in one place.
#
#   T104 Upgrade (BL-E04):
#     slide() now receives objective string so context includes objective reminder.
# ------------------------------------------

class SimConductor:
    def __init__(self, registry):
        self.r = registry

    def run_turn(self, actor):
        """
        Conducts a single turn for a specific actor.
        Returns: (turn_text: str, scene_end: bool)
        turn_text has [[SCENE_END]] stripped — clean for history logging.
        scene_end: True signals T111 loop to stop and archive.
        """

        # ------------------------------------------
        # SECTION 1: CONTEXT GATHERING
        # ------------------------------------------
        ui = self.r.get('ui_cmd')
        if not ui:
            raise Exception("Registry Error: 'ui_cmd' not found.")

        env = ui.env_box.value
        obj = ui.obj_box.value

        # ------------------------------------------
        # SECTION 2: HISTORY + CONTEXT
        # ------------------------------------------
        history = self.r['state'].state.get('global_history', [])
        # BL-E04: Pass objective to slide() for hybrid context + reminder ----
        context = self.r['slider'].slide(history, objective=obj)

        # ------------------------------------------
        # SECTION 3: PERSONA FROM VAULT
        # ------------------------------------------
        # BL-E02: Vault now injects last_utterances anti-repetition block ----
        persona = self.r['vault'].get_full_persona(actor, environment=env)

        # ------------------------------------------
        # SECTION 4: BRAIN GENERATION
        # ------------------------------------------
        # BL-E01: generate() now returns (text, urge, tokens, scene_end) ----
        response, urge, tokens, scene_end = self.r['brain'].generate(
            actor, persona, context, env, obj
        )

        # ------------------------------------------
        # SECTION 5: STRIP [[SCENE_END]] FROM DISPLAY
        # ------------------------------------------
        # [[SCENE_END]] is a signal — strip before logging so it never
        # appears in history context or console as dialogue content. --------
        clean_response = self.r['brain'].strip_scene_end(response)

        # ------------------------------------------
        # SECTION 6: TOKEN AUDIT
        # ------------------------------------------
        self.r['sentinel'].audit_turn(actor, tokens)

        # ------------------------------------------
        # SECTION 7: PERSIST TO HISTORY
        # ------------------------------------------
        turn_text = f"{actor}: {clean_response}"
        self.r['state'].log_event(turn_text)

        return turn_text, scene_end

# ------------------------------------------
# IPO CHECK:
# Input:   actor name (str)
# Process: Gather env/obj/history → build persona → brain.generate()
#          → strip scene signal → audit tokens → persist history
# Output:  (turn_text: str, scene_end: bool)
# ------------------------------------------


In [ ]:
# ==========================================
# T115 : GEMINI_PROV
# ==========================================
# VERSION: 2.2.0 | STATUS: Evolved
# ROLE: google-genai Provider — Safe Text Extraction
# Version Remarks: Unchanged from v1.7.1.
#   _safe_extract_text() prevents ValueError on non-text response parts.
#   Envelope pattern guarantees .text is always str, never raises.
# ------------------------------------------

from google import genai as _genai_module

class GeminiProvider:
    def __init__(self, api_key: str, model_name: str, client=None, log_fn=None):
        self.model_id = model_name.replace("models/", "")
        self.log_fn   = log_fn if log_fn else print
        if client is not None:
            self.client = client
            print(f"✅ T115: Reusing T000 Client → {self.model_id}")
        else:
            self.client = _genai_module.Client(api_key=api_key)
            print(f"✅ T115: New Client created → {self.model_id}")

    def set_log_fn(self, fn):
        self.log_fn = fn

    def _safe_extract_text(self, response) -> str:
        # Path 1: fast path -------------------------------------------
        try:
            text = response.text
            if text: return text
        except (ValueError, AttributeError):
            pass
        # Path 2: walk candidates manually ----------------------------
        try:
            parts = response.candidates[0].content.parts
            collected = [getattr(p, "text", None) for p in parts]
            text = "\n".join(t for t in collected if t)
            if text: return text
        except (IndexError, AttributeError, TypeError):
            pass
        # Path 3: last resort -----------------------------------------
        self.log_fn("⚠️ T115: Could not extract text from response. Check candidates.")
        return ""

    def generate_content(self, prompt: str):
        try:
            raw = self.client.models.generate_content(
                model=self.model_id, contents=prompt)
            usage     = getattr(raw, "usage_metadata",
                                type("U",(),{"prompt_token_count":0,"candidates_token_count":0})())
            safe_text = self._safe_extract_text(raw)

            class Envelope: pass
            env = Envelope()
            env.text           = safe_text
            env.candidates     = getattr(raw, "candidates", [])
            env.usage_metadata = usage
            env._raw           = raw
            return env, usage

        except Exception as e:
            self.log_fn(f"❌ T115 Call Error: {type(e).__name__}: {e}")
            class FailShield: pass
            f = FailShield()
            f.text = ""; f.candidates = []
            u = type("U",(),{"prompt_token_count":0,"candidates_token_count":0})()
            return f, u

# ------------------------------------------
# IPO CHECK:
# Input:   prompt string
# Process: client.models.generate_content() → safe text extraction → Envelope wrap
# Output:  (Envelope with .text str, usage_metadata) — consumed by T102
# ------------------------------------------


In [ ]:
# ==========================================
# T116 : SMOKE_TESTER
# ==========================================
# VERSION: 1.2.2 | STATUS: Stable
# ROLE: End-to-End System Integrity Validation (BFT)
# Version Remarks: Updated _test_state() to use correct StateEngine API.
#                  Updated _test_api() to match T106 new 4-return signature.
# ------------------------------------------

class SmokeTester:
    def __init__(self, registry):
        self.r = registry

    def run_bft(self):
        print("🧪 T116: INITIALIZING FULL SYSTEM BFT...")
        results = {
            "Registry Check": self._test_registry(),
            "State Write":    self._test_state(),
            "API Handshake":  self._test_api()
        }
        print("\n--- BFT REPORT ---")
        all_passed = True
        for test, passed in results.items():
            status = "✅" if passed else "❌"
            print(f"{status} {test}: {'PASSED' if passed else 'FAILED'}")
            if not passed: all_passed = False
        if not all_passed:
            print("\n🚨 CRITICAL: BFT FAILED. DO NOT PROCEED TO UI.")
        return all_passed

    def _test_registry(self):
        keys = ['state', 'brain', 'gateway', 'shield', 'sentinel', 'slider', 'vault']
        missing = [k for k in keys if k not in self.r]
        if missing:
            print(f"   Missing registry keys: {missing}")
            return False
        return True

    def _test_state(self):
        try:
            souls = self.r['state'].load_souls()
            self.r['state'].save_souls(souls)  # round-trip
            return True
        except Exception as e:
            print(f"   [STATE ERROR]: {e}")
            return False

    def _test_api(self):
        try:
            # T106.generate() now returns 4 values — unpack correctly ------
            txt, urge, tokens, scene_end = self.r['brain'].generate(
                "System", "Test actor", "Test context", "Test env", "Ping"
            )
            return bool(txt and txt.strip())
        except Exception as e:
            print(f"   [API ERROR]: {e}")
            return False

# ------------------------------------------
# IPO CHECK:
# Input:   registry (populated)
# Process: Test registry keys → state round-trip → live API ping
# Output:  all_passed bool + printed BFT report
# ------------------------------------------


In [ ]:
# ==========================================
# T117 : UI_CHASSIS
# ==========================================
# VERSION: 5.1.3 | STATUS: Evolving
# ROLE: Master Layout & Tab Management
# Version Remarks: Unchanged from v1.7.1.
# ------------------------------------------

import ipywidgets as widgets
from IPython.display import display

class UIChassis:
    def __init__(self, sov_tab, oracle_tab, metrics_tab):
        self.sov     = sov_tab
        self.oracle  = oracle_tab
        self.metrics = metrics_tab
        self.tab     = None

    def assemble(self):
        self.tab = widgets.Tab()
        self.tab.children = [
            self.sov.get_render_box(),
            self.oracle.get_render_box(),
            self.metrics.get_render_box()
        ]
        self.tab.set_title(0, '🎮 Command Center')
        self.tab.set_title(1, '👁️ Oracle')
        self.tab.set_title(2, '📊 Metrics')

    def display(self):
        if self.tab: display(self.tab)

    def render(self):
        pass

# ------------------------------------------
# IPO CHECK:
# Input:   CommandCenter, UIOracleTab, UIMetrics instances
# Process: Wrap in Tab widget, assign titles
# Output:  Rendered tabbed UI (display() call)
# ------------------------------------------


In [ ]:
# ==========================================
# NOAH'S ARK v1.8.0: MASTER ASSEMBLY — Sprint 1
# ==========================================
# Sprint 1 — Chapter 2.2 — Scene Intelligence & Creator Experience
# Build order: T118 → T000 → T100 → T107 → T102 → T115 → T103
#            → T104 → T108 → T109 → T106 → T114 → T111 → T112
#            → T113 → T110 → T117 → Wire → Observe → Launch
# ------------------------------------------

# ── T118 + T000: Boot ─────────────────────────────────────────────────────────
env_loader = EnvLoader()
provider, model_name, genai_client = boot_sequence(env_loader)
api_key = env_loader.get("GEMINI_API_KEY") or env_loader.get("AGISK_Default")
initialize_env(provider)

# ── Core State + Shield + Provider ────────────────────────────────────────────
state    = StateEngine()
shield   = QuotaShield()
gateway  = GeminiProvider(api_key, model_name, client=genai_client)
vault    = IdentityVault(state)

# ── Processing Tiles ──────────────────────────────────────────────────────────
sentinel  = TokenSentinel()
archiver  = LogArchiver()
slider    = ContextSlider()    # BL-E04: pinned + sliding hybrid
maya      = MayaMetaObserver(gateway, shield)

# ── Registry Bootstrap ────────────────────────────────────────────────────────
registry  = {}
brain     = CharacterBrain(gateway, shield, sentinel, registry)
conductor = SimConductor(registry)

registry.update({
    'state':     state,
    'shield':    shield,
    'gateway':   gateway,
    'vault':     vault,
    'sentinel':  sentinel,
    'archiver':  archiver,
    'slider':    slider,
    'brain':     brain,
    'maya':      maya,
    'conductor': conductor,
})

# ── UI Components ─────────────────────────────────────────────────────────────
ui_cmd    = CommandCenter(registry)
ui_oracle = UIOracleTab()
ui_met    = UIMetrics()

registry['ui_cmd']    = ui_cmd
registry['conductor'] = conductor

# ── Link UI to Logic ──────────────────────────────────────────────────────────
oracle_logic        = OracleLogic(registry)
registry['oracle']  = oracle_logic
ui_oracle.r         = registry
ui_met.sentinel_handle = sentinel

# ── Final Assembly (T117) ─────────────────────────────────────────────────────
chassis = UIChassis(ui_cmd, ui_oracle, ui_met)
chassis.assemble()

# ── Global Tab Observer ───────────────────────────────────────────────────────
def system_observer(change):
    if change['new'] == 2:
        ui_met.refresh_data()
    elif change['new'] == 0:
        if hasattr(ui_cmd, '_refresh_cockpit'):
            ui_cmd._refresh_cockpit()

chassis.tab.observe(system_observer, names='selected_index')

# ── Wire Console Logging (T102 + T115 → Textarea) ─────────────────────────────
ui_cmd.wire_logging()

# 🚀 LAUNCH
chassis.display()
